In [1]:
"""
XGBoost Regressor — Training Approaches
========================================
แสดงวิธีการ train XGBoost Regressor แบบต่างๆ 5 วิธี:
 
  1. Basic fit          — เริ่มต้น, ไม่ปรับ hyperparameter
  2. Early Stopping     — หยุดเทรนอัตโนมัติเมื่อ validation ไม่ดีขึ้น
  3. RandomizedSearchCV — ค้นหา hyperparameter แบบ random grid
  4. Optuna             — ค้นหา hyperparameter แบบ Bayesian (แนะนำ)
  5. xgb.cv()           — cross-validation ด้วย native XGBoost API
 
ข้อมูลในตัวอย่างนี้คือ factory sensor RUL ที่ผ่าน preprocess.py แล้ว
ถ้ายังไม่มี → สร้าง dummy data ให้อัตโนมัติเพื่อรันทดสอบ
"""
 
import warnings
warnings.filterwarnings("ignore")
 
import os
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
 
# ──────────────────────────────────────────────────────────────
# CONFIG — ชี้ไปยังไฟล์ที่ preprocess.py สร้างไว้
# ──────────────────────────────────────────────────────────────
TOROOTPATH  = "../../../model/"
DATA_DIR    = TOROOTPATH + "data/process/factory_sensor/"
X_TRAIN_PATH = DATA_DIR + "X_train.csv"
X_TEST_PATH  = DATA_DIR + "X_test.csv"
Y_TRAIN_PATH = DATA_DIR + "y_train.csv"
Y_TEST_PATH  = DATA_DIR + "y_test.csv"
 
MODEL_OUT_DIR = TOROOTPATH + "models/"
 
TARGET = "Remaining_Useful_Life_days"
 
 
def section(title: str) -> None:
    print(f"\n{'═' * 60}")
    print(f"  {title}")
    print(f"{'═' * 60}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# LOAD DATA  (สร้าง dummy ถ้ายังไม่มีไฟล์)
# ──────────────────────────────────────────────────────────────
def load_data():
    if os.path.exists(X_TRAIN_PATH):
        X_train = pd.read_csv(X_TRAIN_PATH)
        X_test  = pd.read_csv(X_TEST_PATH)
        y_train = pd.read_csv(Y_TRAIN_PATH).squeeze()
        y_test  = pd.read_csv(Y_TEST_PATH).squeeze()
        print(f"  Loaded from {DATA_DIR}")
    else:
        # สร้าง dummy ให้ run ได้โดยไม่ต้องมีไฟล์จริง
        from sklearn.datasets import make_regression
        X, y = make_regression(n_samples=3000, n_features=30,
                               n_informative=20, noise=50, random_state=42)
        X = pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])])
        y = pd.Series(np.clip(y + 500, 0, 1000), name=TARGET)
        from sklearn.model_selection import train_test_split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42)
        print("  (ใช้ dummy dataset — ไม่พบไฟล์จาก preprocess.py)")
    print(f"  Train: {X_train.shape}   Test: {X_test.shape}")
    return X_train, X_test, y_train, y_test

def evaluate(label: str, y_true, y_pred) -> dict:
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"  [{label}]  RMSE={rmse:.2f}  MAE={mae:.2f}  R²={r2:.4f}")
    return {"rmse": rmse, "mae": mae, "r2": r2}

# ══════════════════════════════════════════════════════════════
#  1. BASIC FIT
#     วิธีตรงที่สุด: กำหนด hyperparameter ตายตัว แล้ว fit เลย
#     ใช้เป็น baseline ก่อนเสมอ
# ══════════════════════════════════════════════════════════════
def approach_1_basic(X_train, X_test, y_train, y_test):
    section("1. Basic Fit (baseline)")
 
    model = xgb.XGBRegressor(
        n_estimators   = 300,
        max_depth       = 6,
        learning_rate   = 0.1,
        subsample       = 0.8,
        colsample_bytree= 0.8,
        random_state    = 42,
        n_jobs          = -1,
        verbosity       = 0,
    )
    model.fit(X_train, y_train)
 
    evaluate("train", y_train, model.predict(X_train))
    evaluate("test ", y_test,  model.predict(X_test))
    return model
 
 
# ══════════════════════════════════════════════════════════════
#  2. EARLY STOPPING
#     หยุดเพิ่ม tree อัตโนมัติเมื่อ val RMSE ไม่ดีขึ้นติดต่อกัน N รอบ
#     ป้องกัน overfitting โดยไม่ต้องกำหนด n_estimators ล่วงหน้า
# ══════════════════════════════════════════════════════════════
def approach_2_early_stopping(X_train, X_test, y_train, y_test):
    section("2. Early Stopping")
 
    model = xgb.XGBRegressor(
        n_estimators    = 3000,        # ตั้งสูงไว้ แต่จะหยุดเองก่อน
        max_depth        = 6,
        learning_rate    = 0.05,       # learning rate ต่ำ → ต้องการ round มากกว่า
        subsample        = 0.8,
        colsample_bytree = 0.8,
        random_state     = 42,
        n_jobs           = -1,
        verbosity        = 0,
        early_stopping_rounds = 50,    # หยุดถ้า val ไม่ดีขึ้น 50 รอบติด
        eval_metric       = "rmse",
    )
    model.fit(
        X_train, y_train,
        eval_set    = [(X_train, y_train), (X_test, y_test)],
        verbose     = 100,             # print ทุก 100 round
    )
 
    print(f"\n  Best iteration : {model.best_iteration}")
    print(f"  Best val RMSE  : {model.best_score:.4f}")
    evaluate("train", y_train, model.predict(X_train))
    evaluate("test ", y_test,  model.predict(X_test))
    return model
 
 
# ══════════════════════════════════════════════════════════════
#  3. RandomizedSearchCV
#     สุ่มชุด hyperparameter ใน grid แล้ว cross-validate
#     เหมาะเมื่อ grid ใหญ่ แต่ budget เวลาจำกัด
# ══════════════════════════════════════════════════════════════
def approach_3_randomized_search(X_train, X_test, y_train, y_test):
    section("3. RandomizedSearchCV")
 
    from sklearn.model_selection import RandomizedSearchCV
 
    param_dist = {
        "n_estimators"    : [200, 400, 600, 800],
        "max_depth"        : [3, 4, 5, 6, 7, 8],
        "learning_rate"    : [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
        "subsample"        : [0.6, 0.7, 0.8, 0.9, 1.0],
        "colsample_bytree" : [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        "min_child_weight" : [1, 3, 5, 7],
        "reg_alpha"        : [0, 0.01, 0.1, 1],    # L1 regularization
        "reg_lambda"       : [0.5, 1, 2, 5],        # L2 regularization
    }
 
    base = xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)
 
    search = RandomizedSearchCV(
        estimator     = base,
        param_distributions = param_dist,
        n_iter        = 30,            # ลองกี่ชุด (เพิ่มเพื่อ accuracy มากขึ้น)
        scoring       = "neg_root_mean_squared_error",
        cv            = 5,
        random_state  = 42,
        n_jobs        = -1,
        verbose       = 0,
    )
    search.fit(X_train, y_train)
 
    print(f"  Best params: {search.best_params_}")
    print(f"  Best CV RMSE: {-search.best_score_:.4f}")
    best = search.best_estimator_
    evaluate("train", y_train, best.predict(X_train))
    evaluate("test ", y_test,  best.predict(X_test))
    return best
 
 
# ══════════════════════════════════════════════════════════════
#  4. Optuna (Bayesian / TPE search)  ← แนะนำมากที่สุด
#     Optuna ไม่ได้สุ่ม — ใช้ผลรอบก่อนเลือก hyperparameter รอบต่อไป
#     ค้นหาได้ดีกว่า RandomizedSearch ด้วย trial น้อยกว่า
# ══════════════════════════════════════════════════════════════
def approach_4_optuna(X_train, X_test, y_train, y_test):
    section("4. Optuna (Bayesian Search — แนะนำ)")
 
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    from sklearn.model_selection import cross_val_score
 
    def objective(trial):
        params = {
            "n_estimators"    : trial.suggest_int("n_estimators", 200, 1000),
            "max_depth"        : trial.suggest_int("max_depth", 3, 8),
            "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample"        : trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight" : trial.suggest_int("min_child_weight", 1, 10),
            "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
            "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
        }
        model = xgb.XGBRegressor(**params, random_state=42, n_jobs=-1, verbosity=0)
        scores = cross_val_score(
            model, X_train, y_train,
            cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1,
        )
        return -scores.mean()   # Optuna minimize → คืน RMSE (บวก)
 
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=40, show_progress_bar=True)
 
    print(f"\n  Best trial RMSE: {study.best_value:.4f}")
    print(f"  Best params: {study.best_params}")
 
    # เทรนโมเดลสุดท้ายด้วย best params บน data ทั้งหมด
    best_model = xgb.XGBRegressor(
        **study.best_params, random_state=42, n_jobs=-1, verbosity=0
    )
    best_model.fit(X_train, y_train)
    evaluate("train", y_train, best_model.predict(X_train))
    evaluate("test ", y_test,  best_model.predict(X_test))
    return best_model
 
 
# ══════════════════════════════════════════════════════════════
#  5. xgb.cv()  — Native XGBoost Cross-Validation
#     ใช้ DMatrix (format ภายใน XGBoost) แทน DataFrame
#     ได้ learning curve ต่อ round + early stopping ในตัว
#     เหมาะดูว่า n_estimators ควรเป็นเท่าไหร่ก่อนเลือก params
# ══════════════════════════════════════════════════════════════
def approach_5_xgb_cv(X_train, X_test, y_train, y_test):
    section("5. xgb.cv()  — Native Cross-Validation")
 
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest  = xgb.DMatrix(X_test,  label=y_test)
 
    params = {
        "objective"      : "reg:squarederror",
        "eval_metric"    : "rmse",
        "max_depth"       : 6,
        "learning_rate"   : 0.05,
        "subsample"       : 0.8,
        "colsample_bytree": 0.8,
        "seed"            : 42,
    }
 
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round    = 500,
        nfold              = 5,
        early_stopping_rounds = 30,
        verbose_eval       = 100,
        as_pandas          = True,
    )
 
    best_round = max(1, cv_results["test-rmse-mean"].idxmin())  # guard against 0
    best_rmse  = cv_results.loc[best_round, "test-rmse-mean"]
    print(f"\n  Best round: {best_round}  |  CV RMSE: {best_rmse:.4f}")
 
    # เทรนโมเดลสุดท้ายด้วย best round ที่ได้จาก CV
    final = xgb.train(
        params,
        dtrain,
        num_boost_round = best_round,
        verbose_eval    = False,
    )
    y_pred_test = final.predict(dtest)
    evaluate("test ", y_test, y_pred_test)
 
    # แสดง feature importance (top 15)
    print("\n  Top 15 features by gain:")
    fi = final.get_score(importance_type="gain")
    if not fi:
        print("\n  (feature importance empty — model may have too few rounds)")
        return final
    top = sorted(fi.items(), key=lambda x: x[1], reverse=True)[:15]
    max_score = top[0][1]
    for name, score in top:
        bar = "█" * int(score / max_score * 30)
        print(f"    {name:<32s} {bar} {score:.1f}")
    return final

In [ ]:
# ──────────────────────────────────────────────────────────────
# MAIN — รันทีละ approach
# ──────────────────────────────────────────────────────────────
def main():
    print("\n  Loading data...")
    X_train, X_test, y_train, y_test = load_data()
 
    # รันทีละ approach หรือ uncomment เฉพาะที่ต้องการ
    m1 = approach_1_basic(X_train, X_test, y_train, y_test)
    m2 = approach_2_early_stopping(X_train, X_test, y_train, y_test)
    m3 = approach_3_randomized_search(X_train, X_test, y_train, y_test)
    m4 = approach_4_optuna(X_train, X_test, y_train, y_test)
    m5 = approach_5_xgb_cv(X_train, X_test, y_train, y_test)
 
    # บันทึกโมเดล (เลือกอันดีที่สุดเอง)
    os.makedirs(MODEL_OUT_DIR, exist_ok=True)
    m2.save_model(MODEL_OUT_DIR + "xgb_early_stop.json")
    m4.save_model(MODEL_OUT_DIR + "xgb_optuna.json")
    print(f"\n  Models saved → {MODEL_OUT_DIR}")

if __name__ == "__main__":
    main()